# FluidFlow CFD 1.6
Cavity flows (Re 100-1000), heat transport, rectangular geometry, spectral vortex flows, channel, boundary-layer and pipe-network flow.

Upload the repo files (or `git clone` your repo) first, then run the cells top to bottom.

In [ ]:
!pip -q install -r requirements.txt

## Cavity at Re = 100 vs Ghia et al. (1982)

In [ ]:
!python main.py cavity --re 100 --n 65 --out outputs
from IPython.display import Image
Image('outputs/cavity.png')

## Continuation to higher Reynolds numbers
Warm-starting from the previous solution makes Re = 400 and 1000 much cheaper.

In [ ]:
import main as fl
import numpy as np

cav = fl.LidDrivenCavity(97, 100)
for re in (100, 400, 1000):
    cav.set_reynolds(re)
    info = cav.run(tol=1e-5, max_time=400)
    print(re, info['converged'], 'vortex centre / psi_min:', cav.vortex_centre(), 'Ghia RMS:', round(fl.ghia_error(cav)['rms'], 4))

## Grid convergence (observed order should be about 2)

In [ ]:
res = fl.cavity_convergence(100.0, (17, 33, 65))
for r in res['rows']:
    print(r)
print('observed order:', res['observed_order'])

## Shear layer and pipe flow

In [ ]:
!python main.py shear --out outputs
!python main.py pipe --diameter 0.05 --length 100 --velocity 2 --minor-k 2.5 --efficiency 0.7

In [ ]:
Image('outputs/shear_layer.png')

## Heat transport: differentially heated cavity vs de Vahl Davis (1983)

In [ ]:
for ra, n in ((1e3, 33), (1e4, 41), (1e5, 65)):
    cav = fl.NaturalConvectionCavity(n, ra)
    cav.run()
    print(f'Ra={ra:g}: Nu={cav.nusselt():.3f} (reference {fl.DE_VAHL_DAVIS_NU[ra]}), u_max, v_max = {cav.max_velocities()}')

## Non-square geometry
Wide (W/H = 2) heated cavity and a tall (W/H = 0.5) lid-driven cavity.

In [ ]:
import matplotlib.pyplot as plt
wide = fl.NaturalConvectionCavity(41, 1e4, aspect=2.0)
wide.run()
u, v = wide.velocity()
plt.figure(figsize=(9, 4.5))
plt.contourf(wide.x, wide.y, wide.T, 24, cmap='inferno')
plt.streamplot(wide.x, wide.y, u, v, color='white')
plt.gca().set_aspect('equal'); plt.title(f'Ra=1e4, W/H=2, Nu={wide.nusselt():.3f}'); plt.show()

tall = fl.LidDrivenCavity(49, 100, aspect=0.5)
print(tall.run(tol=1e-5), tall.vortex_centre())

## Boundary layer: Blasius flat plate
Similarity solution by shooting; f''(0) should be 0.33206.

In [ ]:
bl = fl.blasius()
print(bl['fpp0'], bl['delta99'], bl['delta_star'], bl['theta'])
print(fl.flat_plate(5.0, 1.5e-5, 0.5, length=1.0))

## Pipe network (global gradient algorithm)
A looped network with one reservoir and three demand nodes. Flows can reverse (pipe P6).

In [ ]:
e = fl.EXAMPLE_NETWORK
res = fl.pipe_network(e['pipes'], e['fixed_heads'], e['demands'])
print(res['converged'], res['iterations'], res['mass_residual_m3_s'])
for p in res['pipes']:
    print(p['name'], round(p['flow_m3_s'] * 1e3, 3), 'L/s', round(p['head_loss_m'], 3), 'm')
print(res['heads_m'])

## Pump operating point and open-channel flow

In [ ]:
print(fl.pump_operating_point(40, 0.01, 30, 0.1, 200, static_head=10))
print(fl.open_channel(5.0, 3.0, 0.001, 0.013))

## Backwater profile and hydraulic jump

In [ ]:
p = fl.gvf_profile(5.0, 3.0, 0.001, 0.013, y_control=2.0, length=3000.0)
print(p['kind'], p['depth_m'][0], p['depth_m'][-1], p['normal_depth_m'])
print(fl.hydraulic_jump(5.0, 3.0, 0.4))

## 3D views, meshes, models and comparison tables

In [ ]:
!python make_3d.py blasius pump network-model channel-model mesh
from IPython.display import Image, display
for f in ('mesh_3d_cavity_refinement.png', 'model_3d_pipe_network.png', 'model_3d_channel_backwater.png'):
    display(Image(f))